In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict
)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [ ]:
print("Loading MNIST...")

mnist = fetch_openml(
    'mnist_784',
    version=1,
    as_frame=False
)

X = mnist.data.astype(np.float32)
y = mnist.target.astype(int)

In [ ]:
X

In [ ]:
y

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_test_scaled

In [ ]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': [ 'saga'],
    'max_iter': [50]
}

In [ ]:
n_classes = 10

meta_train = np.zeros(
    (X_train_scaled.shape[0], n_classes)
)

meta_test = np.zeros(
    (X_test_scaled.shape[0], n_classes)
)

best_models = {}
best_params = {}

In [ ]:
outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for digit in range(10):

    print("\n" + "="*60)
    print(f"Training model for Digit {digit} vs Rest")
    print("="*60)

    y_binary = (y_train == digit).astype(int)

    # Hyperparameter tuning
    grid = GridSearchCV(
        LogisticRegression(),
        param_grid,
        cv=3,
        scoring='accuracy',
        n_jobs=3
    )

    grid.fit(X_train_scaled, y_binary)

    best_model = grid.best_estimator_

    best_models[digit] = best_model
    best_params[digit] = grid.best_params_

    print("Best Params:", grid.best_params_)

     
    # OOF Predictions (for stacking)
    oof_probs = cross_val_predict(
        best_model,
        X_train_scaled,
        y_binary,
        cv=outer_cv,
        method="predict_proba",
        n_jobs=3
    )[:, 1]

    meta_train[:, digit] = oof_probs

    # Train final model
    best_model.fit(X_train_scaled, y_binary)

    # Binary accuracy
    y_train_pred = best_model.predict(X_train_scaled)

    binary_acc = accuracy_score(y_binary, y_train_pred)

    print("Binary Accuracy:", binary_acc)
    # Test probabilities
    test_probs = best_model.predict_proba(X_test_scaled)[:, 1]
    meta_test[:, digit] = test_probs


Training model for Digit 0 vs Rest


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
print("Meta train shape:", meta_train.shape)
print("Meta test shape:", meta_test.shape)

In [ ]:
meta_model = LogisticRegression( solver='saga', max_iter=1000, tol=0.1)
meta_model.fit(meta_train, y_train)

In [ ]:
y_pred = meta_model.predict(meta_test)

In [ ]:
y_pred

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

plt.title("Stacked Logistic Regression - MNIST")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
for i in range(10):
    print(f"Image {i} → Predicted Digit: {y_pred[i]}")

In [ ]:
import matplotlib.pyplot as plt

for i in range(5):
    plt.imshow(X_test[i].reshape(28,28), cmap='gray')
    plt.title(f"Predicted: {y_pred[i]}")
    plt.axis("off")
    plt.show()

In [ ]:
import numpy as np

def predict_digit_with_confidence(
    image_28x28,
    scaler,
    best_models,
    meta_model
):
    """
    Predict digit and confidence using stacked model.
    """

    # Flatten image
    x = image_28x28.reshape(1, -1).astype(np.float32)
   
     
    # Scale
    x_scaled = scaler.transform(x)

    # Generate meta-features
    meta_features = np.zeros((1, 10))

    for digit in range(10):
        prob = best_models[digit].predict_proba(x_scaled)[0, 1]
        meta_features[0, digit] = prob

    # Final prediction probabilities
    class_probs = meta_model.predict_proba(meta_features)[0]

    predicted_digit = np.argmax(class_probs)
    confidence = class_probs[predicted_digit]


    return {
        "digit": int(predicted_digit),
        "confidence": float(confidence),
        "all_probabilities": class_probs
    }

In [ ]:
from PIL import Image
import numpy as np

def predict_digit_from_file(
    image_path,
    scaler,
    best_models,
    meta_model
):
    img = Image.open(image_path).convert("L")

    img = img.resize((28, 28))

    img_array = np.array(img)

    # Optional: invert colors if digit is black on white background
    img_array = 255 - img_array

    plt.figure(figsize=(3, 3))
    plt.imshow(img_array, cmap="gray")
    plt.title("Model Input (After Inversion)")
    plt.axis("off")
    plt.show()

    pred = predict_digit_with_confidence(
        img_array,
        scaler,
        best_models,
        meta_model
    )

    return pred

In [ ]:
image_array = [
               "/Users/gazifayazwani/Desktop/ML/MNIST/images/two.png", 
               "/Users/gazifayazwani/Desktop/ML/MNIST/images/two2.png", 
               "/Users/gazifayazwani/Desktop/ML/MNIST/images/five.png",
               "/Users/gazifayazwani/Desktop/ML/MNIST/images/five2.png",
               "/Users/gazifayazwani/Desktop/ML/MNIST/images/six.jpeg",
               "/Users/gazifayazwani/Desktop/ML/MNIST/images/eight.png",
               "/Users/gazifayazwani/Desktop/ML/MNIST/images/nine.jpg",
]

for img in image_array:
    pred = predict_digit_from_file(img,
    scaler,
    best_models,
    meta_model
    ) 
    print("Predicted Digit:", pred)

In [ ]:
#minst_data

image_array = [
               "/Users/gazifayazwani/Desktop/ML/MNIST/images/blk_background/eight.jpg",
               "/Users/gazifayazwani/Desktop/ML/MNIST/images/blk_background/five.jpg"
]

for img in image_array:
    pred = predict_digit_from_file(img,
    scaler,
    best_models,
    meta_model
    ) 
    print("Predicted Digit:", pred)